# CSE427 Group 06 - FIFA 23 Player Value Prediction

Cleaned and fixed notebook. This version removes hard-coded Google Drive dependencies, fixes execution order, avoids undefined variables, and computes model results dynamically.


## 1. Load Dataset


In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_style("whitegrid")

os.makedirs("plots", exist_ok=True)
os.makedirs("processed", exist_ok=True)


In [ ]:
# Put the dataset CSV in the same folder as this notebook, or update DATA_PATH manually.
candidate_paths = [
    "Fifa 23 Players Data.csv",
    "FIFA 23 Players Data.csv",
    "fifa23.csv",
    "fifa23_players.csv",
    "data/Fifa 23 Players Data.csv",
    "../data/Fifa 23 Players Data.csv",
    "data/FIFA 23 Players Data.csv",
    "/content/Fifa 23 Players Data.csv",
    "/content/drive/MyDrive/Fifa 23 Players Data.csv",
]

DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    csv_files = glob.glob("*.csv") + glob.glob("data/*.csv") + glob.glob("../data/*.csv")
    if csv_files:
        DATA_PATH = csv_files[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset CSV not found. Place 'Fifa 23 Players Data.csv' in the notebook folder "
        "or set DATA_PATH to the correct CSV path."
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset info:")
df.info()

print("\nTop missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))


## 2. Exploratory Data Analysis


In [ ]:
def save_current_plot(filename):
    plt.tight_layout()
    plt.savefig(os.path.join("plots", filename), dpi=300, bbox_inches="tight")
    plt.show()

if "Overall" in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df["Overall"], bins=30, kde=True, color="steelblue")
    plt.title("Distribution of Player Overall Ratings")
    plt.xlabel("Overall Rating")
    plt.ylabel("Count")
    save_current_plot("overall_distribution.png")
else:
    print("Column 'Overall' not found; skipped overall distribution plot.")


In [ ]:
key_cols = [
    "Overall", "Potential", "Age", "Value(in Euro)", "Wage(in Euro)",
    "Pace Total", "Shooting Total", "Passing Total",
    "Dribbling Total", "Defending Total", "Physicality Total"
]
available_key_cols = [col for col in key_cols if col in df.columns]

if len(available_key_cols) >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(df[available_key_cols].corr(numeric_only=True), annot=True, fmt=".2f",
                cmap="coolwarm", square=True, linewidths=0.5)
    plt.title("Correlation Heatmap of Key Player Attributes")
    save_current_plot("correlation_heatmap.png")
else:
    print("Not enough numeric key columns found; skipped correlation heatmap.")


In [ ]:
if {"Age", "Overall", "Value(in Euro)"}.issubset(df.columns):
    df_eda = df.copy()
    df_eda["Age Group"] = pd.cut(
        df_eda["Age"],
        bins=[15, 21, 25, 29, 45],
        labels=["<=21", "22-25", "26-29", "30+"]
    )

    plt.figure(figsize=(9, 5))
    for group, color in zip(["<=21", "22-25", "26-29", "30+"],
                            ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]):
        sub = df_eda[df_eda["Age Group"] == group]
        plt.scatter(sub["Overall"], sub["Value(in Euro)"] / 1e6,
                    alpha=0.4, s=15, label=group, color=color)

    plt.title("Player Value vs Overall Rating by Age Group")
    plt.xlabel("Overall Rating")
    plt.ylabel("Market Value (Million Euro)")
    plt.legend(title="Age Group")
    save_current_plot("value_vs_overall_by_age.png")
else:
    print("Required columns for value vs overall plot not found; skipped.")


In [ ]:
if {"Preferred Foot", "Skill Moves"}.issubset(df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    foot_counts = df["Preferred Foot"].value_counts()
    axes[0].pie(foot_counts, labels=foot_counts.index, autopct="%1.1f%%",
                colors=["#4FC3F7", "#EF9A9A"], startangle=90)
    axes[0].set_title("Preferred Foot Distribution")

    sns.countplot(data=df, x="Skill Moves", palette="viridis", ax=axes[1])
    axes[1].set_title("Skill Moves Rating Distribution")
    axes[1].set_xlabel("Skill Moves")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(os.path.join("plots", "preferred_foot_skill_moves.png"), dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Required columns for preferred foot and skill moves plot not found; skipped.")


In [ ]:
if {"Age", "Potential", "Overall"}.issubset(df.columns):
    plt.figure(figsize=(9, 5))
    sc = plt.scatter(df["Age"], df["Potential"], c=df["Overall"],
                     cmap="RdYlGn", alpha=0.5, s=15)
    plt.colorbar(sc, label="Overall Rating")
    plt.title("Age vs Potential Colored by Overall Rating")
    plt.xlabel("Age")
    plt.ylabel("Potential Rating")
    save_current_plot("age_vs_potential.png")
else:
    print("Required columns for age vs potential plot not found; skipped.")


## 3. Data Preprocessing


In [ ]:
def standardize_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return dataframe

df = df_raw.copy()
df = df.drop_duplicates()
df = df.drop(columns=["Known As", "Full Name", "Image Link", "National Team Image Link"], errors="ignore")
df = standardize_columns(df)

print("Shape after duplicates and metadata removal:", df.shape)
print(df.columns.tolist()[:30])


In [ ]:
# Convert currency-like columns to numeric.
currency_cols = ["Valuein_Euro", "Wagein_Euro", "Release_Clause"]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(chr(8364), "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "Valuein_Euro" not in df.columns:
    raise KeyError("Target column 'Valuein_Euro' not found after column cleaning.")

print(df[currency_cols].head() if all(c in df.columns for c in currency_cols) else "Currency conversion completed.")


In [ ]:
# Date and contract features.
if "Joined_On" in df.columns:
    df["Joined_On"] = pd.to_datetime(df["Joined_On"], errors="coerce")
    df["Joined_Year"] = df["Joined_On"].dt.year
    df = df.drop(columns=["Joined_On"])

if "Contract_Until" in df.columns:
    df["Contract_Until"] = pd.to_numeric(df["Contract_Until"], errors="coerce")

# Convert boolean-like columns before general encoding.
if "On_Loan" in df.columns:
    df["On_Loan"] = df["On_Loan"].astype(str).str.upper().isin(["TRUE", "1", "YES"]).astype(int)

if "Preferred_Foot" in df.columns:
    df["Preferred_Foot"] = df["Preferred_Foot"].map({"Right": 1, "Left": 0}).fillna(0).astype(int)

workrate_map = {"Low": 0, "Medium": 1, "High": 2}
for col in ["Attacking_Work_Rate", "Defensive_Work_Rate"]:
    if col in df.columns:
        df[col] = df[col].map(workrate_map)


In [ ]:
# Positions feature engineering.
if "Positions_Played" in df.columns:
    df["Positions_Count"] = df["Positions_Played"].apply(
        lambda x: len(str(x).split(",")) if pd.notna(x) and str(x).strip() else 0
    )
    df = df.drop(columns=["Positions_Played"])

position_cols = [col for col in ["Best_Position", "Club_Position", "National_Team_Position"] if col in df.columns]
if position_cols:
    df = pd.get_dummies(df, columns=position_cols, prefix=position_cols, drop_first=False)

# Remove high-cardinality columns that can cause overfitting and memory issues.
high_card_cols = ["Nationality", "Club_Name", "National_Team_Name"]
df = df.drop(columns=[col for col in high_card_cols if col in df.columns], errors="ignore")


In [ ]:
# Missing value handling.
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if df[col].isnull().any():
        mode_value = df[col].mode()
        fill_value = mode_value.iloc[0] if len(mode_value) else "Unknown"
        df[col] = df[col].fillna(fill_value)

# Encode any remaining categorical columns.
remaining_cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
if remaining_cat_cols:
    df = pd.get_dummies(df, columns=remaining_cat_cols, drop_first=False)

# Convert boolean dummy columns to integers.
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

# Drop remaining rows with missing values, if any.
df = df.dropna().reset_index(drop=True)

print("Final missing values:", int(df.isnull().sum().sum()))
print("Final preprocessed shape:", df.shape)


In [ ]:
# Remove zero targets and apply log transform to reduce skewness.
df = df[df["Valuein_Euro"] > 0].reset_index(drop=True)
df["Valuein_Euro"] = np.log1p(df["Valuein_Euro"])

print("Shape after removing zero-value players:", df.shape)
print("Target skewness after log transform:", df["Valuein_Euro"].skew())

df.to_csv("processed/fifa23_final_ready_cleaned.csv", index=False)
print("Saved cleaned dataset to processed/fifa23_final_ready_cleaned.csv")


## 4. Train-Test Split and Feature Scaling


In [ ]:
X = df.drop(columns=["Valuein_Euro"])
y = df["Valuein_Euro"]

# Ensure all features are numeric.
X = X.select_dtypes(include=[np.number]).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Remove columns that become NaN after scaling, usually constant features.
nan_cols = np.any(np.isnan(X_train_scaled), axis=0)
X_train_clean = X_train_scaled[:, ~nan_cols]
X_test_clean = X_test_scaled[:, ~nan_cols]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Scaled train shape:", X_train_clean.shape)


In [ ]:
# Feature importance after the train-test split.
model_et = ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_et.fit(X_train, y_train)

importances = pd.Series(model_et.feature_importances_, index=X.columns)
top10 = importances.nlargest(10).sort_values()

plt.figure(figsize=(10, 6))
top10.plot(kind="barh", color="steelblue", edgecolor="white")
plt.xlabel("Feature Importance Score")
plt.title("Top 10 Most Important Features (Extra Trees)")
save_current_plot("feature_importance.png")


## 5. Model Training and Evaluation


In [ ]:
results = []

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    rmse = mean_squared_error(y_te, y_pred) ** 0.5
    mae = mean_absolute_error(y_te, y_pred)
    r2 = r2_score(y_te, y_pred)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    print("\n" + "=" * 40)
    print(name)
    print("=" * 40)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R2  : {r2:.4f}")

    return model


In [ ]:
# Linear models use scaled features.
linear_model = evaluate_model("Linear Regression", LinearRegression(), X_train_clean, y_train, X_test_clean, y_test)
ridge_model = evaluate_model("Ridge Regression", Ridge(alpha=1.0, random_state=42), X_train_clean, y_train, X_test_clean, y_test)
lasso_model = evaluate_model("Lasso Regression", Lasso(alpha=0.001, random_state=42, max_iter=10000), X_train_clean, y_train, X_test_clean, y_test)


In [ ]:
# Tree-based models use the original unscaled numeric features.
dt_model = evaluate_model("Decision Tree", DecisionTreeRegressor(random_state=42), X_train, y_train, X_test, y_test)

rf_model = evaluate_model(
    "Random Forest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)

extra_model = evaluate_model(
    "Extra Trees",
    ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)

gb_model = evaluate_model(
    "Gradient Boosting",
    GradientBoostingRegressor(n_estimators=100, random_state=42),
    X_train_clean, y_train, X_test_clean, y_test
)


In [ ]:
# XGBoost is optional. If it is not installed, the notebook skips it cleanly.
try:
    import xgboost as xgb

    xgb_model = evaluate_model(
        "XGBoost",
        xgb.XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        ),
        X_train_clean, y_train, X_test_clean, y_test
    )
except ImportError:
    print("xgboost is not installed; skipped XGBoost model.")


In [ ]:
mlp_model = evaluate_model(
    "MLP",
    MLPRegressor(
        hidden_layer_sizes=(100, 50),
        activation="relu",
        solver="adam",
        max_iter=500,
        random_state=42,
        early_stopping=True
    ),
    X_train_clean, y_train, X_test_clean, y_test
)


## 6. Results Visualization


In [ ]:
df_results = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
df_results.to_csv("processed/model_results.csv", index=False)

print("Model performance comparison:")
print(df_results.to_string(index=False))


In [ ]:
plt.figure(figsize=(10, 6))
colors = ["#2ecc71" if r2 == df_results["R2"].max() else "#3498db" for r2 in df_results["R2"]]
plt.barh(df_results["Model"], df_results["R2"], color=colors, edgecolor="white")
plt.xlabel("R2 Score")
plt.title("Model Performance Comparison (R2 Score)")
plt.gca().invert_yaxis()
save_current_plot("model_comparison_r2.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_rmse = df_results.sort_values("RMSE", ascending=True)
axes[0].barh(df_rmse["Model"], df_rmse["RMSE"], color="#e74c3c", edgecolor="white")
axes[0].set_xlabel("RMSE (lower is better)")
axes[0].set_title("RMSE Comparison")

df_mae = df_results.sort_values("MAE", ascending=True)
axes[1].barh(df_mae["Model"], df_mae["MAE"], color="#f39c12", edgecolor="white")
axes[1].set_xlabel("MAE (lower is better)")
axes[1].set_title("MAE Comparison")

axes[2].barh(df_results["Model"], df_results["R2"], color="#2ecc71", edgecolor="white")
axes[2].set_xlabel("R2 (higher is better)")
axes[2].set_title("R2 Comparison")
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join("plots", "model_comparison_all_metrics.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

best_r2 = df_results.iloc[0]
best_rmse = df_results.loc[df_results["RMSE"].idxmin()]
best_mae = df_results.loc[df_results["MAE"].idxmin()]

print(f"Best model by R2:   {best_r2['Model']} (R2 = {best_r2['R2']:.4f})")
print(f"Best model by RMSE: {best_rmse['Model']} (RMSE = {best_rmse['RMSE']:.4f})")
print(f"Best model by MAE:  {best_mae['Model']} (MAE = {best_mae['MAE']:.4f})")

print("\nModel rankings by R2:")
for i, row in df_results.iterrows():
    print(f"{i + 1}. {row['Model']:<25} R2 = {row['R2']:.4f}")
